# 🌿 Darukaa Reference Benchmarking Pipeline

**Compare any biodiversity indicator against ecoregion-specific reference benchmarks, then compute the State of Nature Score per the State of Nature Module PRD v2.0.**

This notebook runs the full pipeline:
1. Clone the repo & install dependencies
2. Authenticate Google Earth Engine
3. Upload your KML site file
4. Configure and run Tier 1 (regional) + Tier 2 (contemporary) benchmarking
5. View the benchmark scorecard
6. Compute State of Nature concern levels and SoN Score
7. Download results

**Reference benchmarking methodology:**
- McElderry et al. (2024) DOI:10.32942/X2689N — SEED reference selection (adapted)
- McNellie et al. (2020) DOI:10.1111/gcb.15383 — Contemporary reference states
- Yen et al. (2019) DOI:10.1002/eap.1970 — Biodiversity benchmarks

**State of Nature scoring methodology:**
- Darukaa State of Nature Module PRD v2.0
- TNFD LEAP Annex 2 (Ecosystem Extent / Condition / Species Population / Extinction Risk)
- Normalized sum formula: `SoN_Score = (Σ dim_scores − 4) / 16 × 10`

## 1. Setup — Clone repo & install

In [22]:
# ── Setup: Clone or update repo ──────────────────────────
import os
os.chdir("/content")  # Always start from a safe location

REPO_DIR = "/content/reference-benchmarking"
CODE_DIR = f"{REPO_DIR}/darukaa_reference_v0.1.0"

if os.path.exists(REPO_DIR):
    os.chdir(CODE_DIR)
    !git pull origin main
    print("✓ Pulled latest changes")
else:
    !git clone https://@github.com/G-auravSingh/reference-benchmarking.git {REPO_DIR}
    os.chdir(CODE_DIR)
    print("✓ Cloned fresh")

!pip install -e . --force-reinstall -q 2>&1 | tail -1

import sys
cleared = [k for k in sys.modules if 'darukaa' in k]
for m in cleared:
    del sys.modules[m]

os.chdir("/content")
print(f"✓ Installed from {CODE_DIR}, cleared {len(cleared)} cached modules")

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 11 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 3.85 KiB | 563.00 KiB/s, done.
From https://github.com/G-auravSingh/reference-benchmarking
 * branch            main       -> FETCH_HEAD
   29a6992..b9822b5  main       -> origin/main
Updating 29a6992..b9822b5
Fast-forward
 .../darukaa_reference/indicators/__init__.py       | 75 ++++++++++++++++++++--
 .../darukaa_reference/reference.py                 | 11 ++++
 2 files changed, 81 insertions(+), 5 deletions(-)
✓ Pulled latest changes
grpc-google-iam-v1 0.14.3 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
✓ Installed from /content/reference-benchmarking/darukaa_reference_v0.1.0, cleared 10 cached modules


## 2. Authenticate Google Earth Engine

Run the cell below — it will show a link. Click it, sign in with your Google account, copy the token back.

In [23]:
import ee

# Colab has a built-in GEE authenticator
ee.Authenticate()

# Initialize with your GEE project
# Replace with your actual project ID
GEE_PROJECT = "gaurav-singh-007"  # @param {type:"string"}

ee.Initialize(project=GEE_PROJECT)
print(f"✓ GEE initialised with project: {GEE_PROJECT}")

✓ GEE initialised with project: gaurav-singh-007


## 3. Upload your KML file

Run the cell below to upload your project site KML/KMZ/GeoJSON file.

In [3]:
from google.colab import files

print("Upload your KML/KMZ/GeoJSON file:")
uploaded = files.upload()

# Get the filename
SITE_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {SITE_FILE} ({len(uploaded[SITE_FILE])/1024:.1f} KB)")

Upload your KML/KMZ/GeoJSON file:


Saving Paglam_CR_Demarcation2025.kmz to Paglam_CR_Demarcation2025.kmz

✓ Uploaded: Paglam_CR_Demarcation2025.kmz (1.6 KB)




```
# This is formatted as code
```

## 4. Configure the pipeline

Key parameters:
- `HMI_CEILING`: 0.10 (adapted from SEED's 0.05 for buffer-scale analysis — see README)
- `ELEVATION_BAND_M`: ±300m elevation stratification for Tier 2
- `NDVI_YEAR` / `LST_YEAR`: remote sensing year for temporal indicators
- `ENABLED_INDICATORS`: leave empty for all 25 registered indicators

In [24]:
# ── Configuration ─────────────────────────────────────────

# Which indicators to run? Leave empty for all 25.
ENABLED_INDICATORS = []  # e.g. ["ndvi", "bii", "eii"] to run a subset

# Tier 2 reference selection parameters
HMI_CEILING = 0.10       # Adapted from SEED's 0.05 for buffer-scale analysis
ELEVATION_BAND_M = 300.0 # ±m elevation filter for Tier 2 stratification
MIN_REF_PIXELS = 5       # SEED minimum before fallback cascade

# Remote sensing year
NDVI_YEAR = 2025  # @param {type:"integer"}
LST_YEAR  = 2025  # @param {type:"integer"}

# Output
OUTPUT_DIR = "./output"

# ─────────────────────────────────────────────────────────
print(f"HMI ceiling:        {HMI_CEILING}")
print(f"Elevation band:     ±{ELEVATION_BAND_M}m")
print(f"Min ref pixels:     {MIN_REF_PIXELS}")
print(f"Remote sensing year: {NDVI_YEAR}")
print(f"Indicators:         {'all 25' if not ENABLED_INDICATORS else ENABLED_INDICATORS}")

HMI ceiling:        0.1
Elevation band:     ±300.0m
Min ref pixels:     5
Remote sensing year: 2025
Indicators:         all 25


## 5. Run the pipeline

In [25]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
import sys, os
sys.path.append("/content/reference-benchmarking/darukaa_reference_v0.1.0")
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
from darukaa_reference.pipeline import Pipeline

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build config
# Key Darukaa GEE assets — cross-project access required
config = Config(
    gee_project=GEE_PROJECT,
    bii_gee_asset="projects/gaurav-singh-007/assets/bii-2020_v2-1-1",
    hmi_hard_ceiling=HMI_CEILING,
    elevation_band_m=ELEVATION_BAND_M,
    min_reference_pixels=MIN_REF_PIXELS,
    ndvi_year=NDVI_YEAR,
    lst_year=LST_YEAR,
    raster_paths={
        # Darukaa-hosted GEE FeatureCollection assets
        "iucn_mammals": "projects/darukaa-earth130226/assets/RedList_Mammals_Terrestrial",
        "iucn_birds":   "projects/darukaa-earth130226/assets/RedList_Bird_IUCN_Category",
        "kba_global":   "projects/darukaa-earth130226/assets/KBA_Global_POL_SEP25",
        # India-only PV binary for CPLAND
        "pv_binary":    "projects/darukaa-earth-product/assets/biodiversity_India_PV_Binary_2025_Full_Mosaic",
        # MSA (WIP — not yet registered)
        "msa":          "projects/ee-jayankandir/assets/TerrestrialMSA_2015_World",
    },
    output_dir=OUTPUT_DIR,
    output_format="both",
    enabled_indicators=ENABLED_INDICATORS,
)

# Create registry (25 indicators, TNFD Annex 2 aligned)
registry = create_default_registry()
print(f"✓ Registry loaded: {len(registry)} indicators")
print(f"  Tier 2 eligible: {len(registry.tier2_indicators())} indicators")
print(f"  Tier 2 excluded (threats): {len(registry) - len(registry.tier2_indicators())} indicators")

# Build and run pipeline
pipeline = Pipeline(config, registry)
report = pipeline.run(
    site_path=SITE_FILE,
    output_path=f"{OUTPUT_DIR}/benchmark_scorecard",
)
print("\n✓ Pipeline complete")

✓ Registry loaded: 25 indicators
  Tier 2 eligible: 13 indicators
  Tier 2 excluded (threats): 12 indicators

✓ Pipeline complete


## 6. Benchmark Scorecard

Raw output: site values, Tier 1 (regional) and Tier 2 (contemporary least-disturbed) references, and intactness ratios for all indicators.

In [26]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(report["scorecard"])

display_cols = [
    "site_id", "display_name", "pillar", "site_value",
    "tier1_reference", "tier1_intactness",
    "tier2_reference", "tier2_intactness",
    "interpretation",
]
existing_cols = [c for c in display_cols if c in df.columns]

print(f"{'='*70}")
print(f"BENCHMARK SCORECARD: {len(df)} indicator-site pairs")
print(f"{'='*70}\n")

# Format for display
df_display = df[existing_cols].copy()
for col in ["site_value", "tier1_reference", "tier2_reference"]:
    if col in df_display:
        df_display[col] = df_display[col].apply(
            lambda x: f"{x:.4f}" if pd.notna(x) else "—"
        )
for col in ["tier1_intactness", "tier2_intactness"]:
    if col in df_display:
        df_display[col] = df_display[col].apply(
            lambda x: f"{x:.1%}" if pd.notna(x) else "—"
        )

pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 30)
display(df_display)

BENCHMARK SCORECARD: 25 indicator-site pairs



,site_id,display_name,pillar,site_value,tier1_reference,tier1_intactness,tier2_reference,tier2_intactness,interpretation
0,Paglam_CR_Demarcation2025_0000,Natural Habitat Extent,1,95.9952,100.0000,96.0%,100.0000,96.0%,Near-reference condition (96.0% of T...
1,Paglam_CR_Demarcation2025_0000,Natural Land Cover Proportion,1,87.4370,100.0000,87.4%,100.0000,87.4%,Moderate intactness (87.4% of Tier 2...
2,Paglam_CR_Demarcation2025_0000,Landscape Connectivity (CPLAND),1,88.1717,1.0000,100.0%,—,—,Insufficient data for interpretation
3,Paglam_CR_Demarcation2025_0000,Habitat Loss Rate,1,0.2781,4.1667,100.0%,4.1667,100.0%,Near-reference condition (100.0% of ...
4,Paglam_CR_Demarcation2025_0000,KBA/IBA Overlap,1,100.0000,—,—,—,—,Insufficient data for interpretation
5,Paglam_CR_Demarcation2025_0000,Vegetation Structure (NDVI),2,0.6298,0.6751,93.3%,0.8635,72.9%,Moderate intactness (72.9% of Tier 2...
6,Paglam_CR_Demarcation2025_0000,Habitat Health Index (HHI),2,2.4687,4.3157,57.2%,9.8004,25.2%,Severely degraded (25.2% of Tier 2 b...
7,Paglam_CR_Demarcation2025_0000,Forest Landscape Integrity Index,2,7.5685,9.9763,75.9%,9.9758,75.9%,Moderate intactness (75.9% of Tier 2...
8,Paglam_CR_Demarcation2025_0000,Ecosystem Integrity Index,2,0.4678,0.4588,100.0%,0.4823,97.0%,Near-reference condition (97.0% of T...
9,Paglam_CR_Demarcation2025_0000,EII: Structural Integrity,2,0.6007,0.6286,95.5%,0.6053,99.2%,Near-reference condition (99.2% of T...


## 7. State of Nature Scoring

Converts intactness ratios into concern levels and computes the State of Nature Score per **Darukaa SoN Module PRD v2.0** and **TNFD LEAP Annex 2**.

**Three-tier scoring:**
- **Tier 1 (per-metric):** Intactness % → concern level (VL/L/M/H/VH) via threshold protocol
- **Tier 2 (per-dimension):** Mean concern numeric per TNFD dimension
- **Tier 3 (site-level):** `SoN_Score = (Σ dim_scores − n_dims) / (n_dims × 4) × 10` → 0–10 scale

**Protocol B v1.0 — Fixed, version-controlled thresholds (applied to intactness %):**
```
≥ 85% → VL (Very Low)    Basis: Newbold et al. 2016 planetary boundary (~90% BII)
70–84% → L  (Low)        Noticeably impacted, recoverable (SBTN science)
50–69% → M  (Moderate)   Substantially degraded — material loss (GLOBIO4 literature)
30–49% → H  (High)       Severely impaired
<  30% → VH (Very High)  Critically impaired
```
Review trigger: ≥10 Darukaa sites across ≥3 ecoregion types. These thresholds are fixed, not project-specific.

**Protocol A:** BII (NHM thresholds), FLII (Potapov thresholds) — fixed published literature values.

**Dimension scoring:** Computed from all indicators with a populated concern level — **no minimum metric constraints** at this stage. Missing metrics are shown as `n=X of Y` so confidence is transparent. The SoN PRD v2.0 data sufficiency rules will be enforced in the product UI layer once in-situ data is integrated.

**Threats** (gHM, light pollution, HDI, LST) are contextual only — not in SoN score per TNFD Annex 2.

In [27]:
import numpy as np

# ── Threshold Tables ─────────────────────────────────────────────────────
# Numeric encoding: VL=1, L=2, M=3, H=4, VH=5

# Protocol A: Published literature thresholds (applied to raw site value)
# These indicators have defensible absolute thresholds — no intactness ratio needed.
PROTOCOL_A = {
    # ── Ecosystem Condition ──
    # BII: NHM / TNFD Annex 2 (higher=better, 0-1)
    "bii":  {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # FLII: Potapov et al. 2020 (higher=better, 0-10)
    "flii": {"breaks": [2.0,  4.0,  6.0,  8.0],  "scores": [5, 4, 3, 2, 1], "higher_is_better": True},
    # MSA: GLOBIO4 (higher=better, 0-1) -- not yet populated
    "msa":  {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},

    # ── Species Population Size ──
    # Flagship Habitat Viability: HSI literature (higher=better, 0-1)
    # >0.8=VL (excellent), 0.6-0.8=L (good), 0.4-0.6=M (moderate), 0.2-0.4=H (poor), <0.2=VH
    "flagship_habitat": {"breaks": [0.20, 0.40, 0.60, 0.80], "scores": [5, 4, 3, 2, 1], "higher_is_better": True},

    # ── Species Extinction Risk ──
    # CERI: Butchart et al. 2007 (lower=better, 0-1)
    # 0-0.1=VL (LC dominated), 0.1-0.2=L, 0.2-0.35=M, 0.35-0.5=H, >0.5=VH
    "ceri": {"breaks": [0.10, 0.20, 0.35, 0.50], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},

    # STAR_T: IUCN STAR methodology (lower=better, higher score = more abatement needed)
    # 0=VL, 1-3=L, 3-6=M, 6-9=H, >9=VH
    "star_t": {"breaks": [1.0, 3.0, 6.0, 9.0], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},

    # KBA/IBA Overlap: TNFD/IBAT scheme (higher overlap = higher concern for disclosure)
    # Note: KBA site value = % overlap. 0%=VL, 1-25%=L, 25-75%=M, 75-99%=H, 100%=VH
    # Rationale: being inside a KBA signals high biodiversity importance (nature+ opportunity)
    # and high dependency/impact risk — TNFD treats this as elevated concern for disclosure
    "kba_overlap": {"breaks": [1.0, 25.0, 75.0, 99.9], "scores": [1, 2, 3, 4, 5], "higher_is_better": False},
}

# Protocol B v1.0 -- FIXED thresholds (applied to Tier 2 intactness %, or Tier 1 where T2 unavailable)
# Scientific basis:
#   >=85% (VL): Newbold et al. 2016 planetary boundary
#   70-84% (L): Noticeably impacted, recoverable (SBTN)
#   50-69% (M): Substantially degraded (GLOBIO4)
#   30-49% (H): Severely impaired
#    <30% (VH): Critically impaired
# Version: Protocol B v1.0 | Review trigger: >=10 sites across >=3 ecoregion types
PROTOCOL_B_BREAKS = [30, 50, 70, 85]
PROTOCOL_B_SCORES = [5,  4,  3,  2, 1]

# Protocol C: Tier 1 regional comparison — for count/connectivity indicators where
# absolute thresholds are ecoregion-dependent. Intactness = site / T1_regional_mean.
# These use the same Protocol B intactness % thresholds applied to T1 intactness.
# Indicators: natural_habitat*, natural_landcover*, cpland, forest_loss_rate*,
#             endemic_richness, threatened_richness
# (* also have T2, so C only applies when T2 fails)
PROTOCOL_C_INDICATORS = {"cpland", "endemic_richness", "threatened_richness"}

# TNFD Annex 2 dimension membership
DIM_MAP = {
    1: ["natural_habitat", "natural_landcover", "cpland", "forest_loss_rate", "kba_overlap"],
    2: ["ndvi", "habitat_health", "flii", "eii", "eii_structural",
        "eii_compositional", "eii_functional", "bii", "pdf", "aridity_index"],
    3: ["endemic_richness", "flagship_habitat"],
    4: ["threatened_richness", "ceri", "star_t"],
}
THREATS = ["ghm", "light_pollution", "hdi", "lst_day", "lst_night"]
DIM_NAMES = {
    1: "Ecosystem Extent",
    2: "Ecosystem Condition",
    3: "Species Population Size",
    4: "Species Extinction Risk",
}
PROTOCOL_A_INDICATORS = set(PROTOCOL_A.keys())


# ── Helpers ──────────────────────────────────────────────────────────────

def _is_valid(v):
    "True if v is a real finite number (not None, NaN, inf)."
    if v is None: return False
    try: return np.isfinite(float(v))
    except: return False

def apply_protocol_a(indicator_name, site_value):
    "Apply published absolute threshold. Returns concern numeric 1-5 or None."
    if not _is_valid(site_value): return None
    spec = PROTOCOL_A[indicator_name]
    val = float(site_value)
    # For lower-is-better: scores list already inverted (lower val = lower score = VL)
    for i, b in enumerate(spec["breaks"]):
        if val < b: return spec["scores"][i]
    return spec["scores"][-1]

def apply_protocol_b(intactness_ratio):
    "Protocol B v1.0 / C: intactness ratio 0-1 -> concern numeric."
    if not _is_valid(intactness_ratio): return None
    pct = float(intactness_ratio) * 100
    for i, b in enumerate(PROTOCOL_B_BREAKS):
        if pct < b: return PROTOCOL_B_SCORES[i]
    return PROTOCOL_B_SCORES[-1]

def concern_label(numeric):
    if numeric is None: return "—"
    return {1: "VL", 2: "L", 3: "M", 4: "H", 5: "VH"}.get(round(numeric), "—")

def dim_label(score):
    if score is None: return "—"
    s = round(score * 2) / 2
    if s <= 1.5: return "Very Low"
    if s <= 2.5: return "Low"
    if s <= 3.5: return "Moderate"
    if s <= 4.5: return "High"
    return "Very High"


# ── Build results lookup ──────────────────────────────────────────────────
results_lookup = {}
for _, row in df.iterrows():
    name = row.get("indicator_name") or row.get("name")
    if name is None:
        for spec in registry.all():
            if spec.display_name == row.get("display_name"):
                name = spec.name; break
    if name:
        results_lookup[name] = row.to_dict()


# ── Per-metric concern assignment ────────────────────────────────────────
# Priority order per indicator:
#   1. Protocol A (published absolute threshold) — BII, FLII, CERI, KBA, Flagship, STAR_T
#   2. Protocol B v1.0 on Tier 2 intactness    — all other indicators with T2
#   3. Protocol C (= B thresholds on Tier 1)   — CPLAND, endemic, threatened when T2 absent
#   4. Protocol B on Tier 1 fallback            — any remaining indicator with T1
#   5. No reference                             — concern = None, reported as "no ref"

metric_concerns = {}

for name, row in results_lookup.items():
    if name in THREATS:
        continue

    site_val = row.get("site_value")
    t2       = row.get("tier2_intactness")
    t1       = row.get("tier1_intactness")

    if not _is_valid(site_val): site_val = None
    if not _is_valid(t2):       t2 = None
    if not _is_valid(t1):       t1 = None

    spec_obj = registry.get(name) if name in registry else None
    hib = spec_obj.higher_is_better if spec_obj else True

    if name in PROTOCOL_A_INDICATORS and site_val is not None:
        # Protocol A: published absolute threshold
        cn = apply_protocol_a(name, site_val)
        protocol = "A (published absolute threshold)"
        intactness_used = None
        t_ref = None

    elif t2 is not None:
        # Protocol B v1.0: Tier 2 intactness (preferred)
        cn = apply_protocol_b(t2)
        protocol = "B v1.0 (Tier 2 intactness)"
        intactness_used = t2
        t_ref = row.get("tier2_reference")

    elif t1 is not None:
        # Protocol B/C on Tier 1: designed reference for count/connectivity indicators,
        # or fallback for raster indicators where T2 failed
        cn = apply_protocol_b(t1)
        if name in PROTOCOL_C_INDICATORS:
            protocol = "C (Tier 1 regional — designed ref for this indicator)"
        else:
            protocol = "B v1.0 (Tier 1 fallback — T2 unavailable)"
        intactness_used = t1
        t_ref = row.get("tier1_reference")

    else:
        # Genuinely no reference of any kind
        cn = None
        protocol = "— (no reference computable)"
        intactness_used = None
        t_ref = None

    metric_concerns[name] = {
        "concern_numeric":  cn,
        "concern_label":    concern_label(cn),
        "protocol":         protocol,
        "intactness_used":  intactness_used,
        "site_value":       site_val,
        "display_name":     row.get("display_name", name),
    }

n_scored  = sum(1 for v in metric_concerns.values() if v["concern_numeric"] is not None)
n_nodata  = sum(1 for v in metric_concerns.values() if v["concern_numeric"] is None)
n_prot_a  = sum(1 for v in metric_concerns.values() if "A" in v["protocol"])
n_prot_b  = sum(1 for v in metric_concerns.values() if "B v1.0" in v["protocol"] and "C" not in v["protocol"])
n_prot_c  = sum(1 for v in metric_concerns.values() if "C" in v["protocol"])
print(f"\u2713 Per-metric concern levels assigned")
print(f"  Protocol A (published):    {n_prot_a}")
print(f"  Protocol B v1.0 (T2):      {n_prot_b}")
print(f"  Protocol C / B-T1:         {n_prot_c}")
print(f"  No reference:              {n_nodata}")
print(f"  Total scored:              {n_scored} / {len(metric_concerns)}")


✓ Per-metric concern levels assigned
  Protocol A (published):    5
  Protocol B v1.0 (T2):      11
  Protocol C / B-T1:         3
  No reference:              1
  Total scored:              19 / 20


In [28]:
# ── Per-metric concern table ─────────────────────────────────────────────
print(f"\n{'='*92}")
print(f"PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0)")
print(f"{'='*92}")
print(f"  {'Dim':<5} {'Indicator':<42} {'Site Value':<12} {'Intactness':<12} {'Concern':<8} Protocol")
print("  " + "-"*90)

for dim_num, indicators in DIM_MAP.items():
    print(f"\n  ── Dim {dim_num}: {DIM_NAMES[dim_num]} ──")
    for name in indicators:
        mc  = metric_concerns.get(name)
        row = results_lookup.get(name, {})
        dname = (mc["display_name"] if mc else row.get("display_name", name))[:41]

        if mc is None:
            print(f"  {dim_num:<5} {dname:<42} {'—':<12} {'—':<12} {'—':<8} not run")
            continue

        sv = mc["site_value"]
        sv_str = f"{sv:.4f}" if sv is not None else "—"

        iu = mc["intactness_used"]
        if iu is not None:
            intact_str = f"{iu:.1%}"
        elif mc["protocol"] == "A (published)":
            intact_str = "n/a (raw)"
        else:
            intact_str = "no ref"  # site value exists but no benchmark

        print(f"  {dim_num:<5} {dname:<42} {sv_str:<12} {intact_str:<12} {mc['concern_label']:<8} {mc['protocol']}")

print(f"\n  ── Threats (contextual — not in SoN score) ──")
for name in THREATS:
    row   = results_lookup.get(name, {})
    dname = row.get("display_name", name)[:41]
    sv    = row.get("site_value")
    t1    = row.get("tier1_intactness")
    sv_str = f"{float(sv):.4f}" if _is_valid(sv) else "—"
    t1_str = f"{float(t1):.1%}"  if _is_valid(t1) else "—"
    print(f"  {'T':<5} {dname:<42} {sv_str:<12} {t1_str:<12} {'—':<8} Tier 1 contextual")



PER-METRIC CONCERN LEVELS  (TNFD Annex 2 | Protocol B v1.0)
  Dim   Indicator                                  Site Value   Intactness   Concern  Protocol
  ------------------------------------------------------------------------------------------

  ── Dim 1: Ecosystem Extent ──
  1     Natural Habitat Extent                     95.9952      96.0%        VL       B v1.0 (Tier 2 intactness)
  1     Natural Land Cover Proportion              87.4370      87.4%        VL       B v1.0 (Tier 2 intactness)
  1     Landscape Connectivity (CPLAND)            88.1717      100.0%       VL       C (Tier 1 regional — designed ref for this indicator)
  1     Habitat Loss Rate                          0.2781       100.0%       VL       B v1.0 (Tier 2 intactness)
  1     KBA/IBA Overlap                            100.0000     no ref       VH       A (published absolute threshold)

  ── Dim 2: Ecosystem Condition ──
  2     Vegetation Structure (NDVI)                0.6298       72.9%        L      

In [30]:
# ── Dimension scores & SoN Score ─────────────────────────────────────────
# Dimension score = mean concern numeric of all indicators with populated data.
# No minimum metric constraints — partial coverage shown via n_pop/n_total.
# Data sufficiency rules (SoN PRD v2.0) enforced in product UI, not here.

dim_scores   = {}
dim_populated = {}

for dim_num, indicators in DIM_MAP.items():
    values, populated_names = [], []
    for name in indicators:
        mc = metric_concerns.get(name)
        if mc and mc["concern_numeric"] is not None:
            values.append(mc["concern_numeric"])
            populated_names.append(name)
    dim_populated[dim_num] = populated_names
    dim_scores[dim_num] = sum(values) / len(values) if values else None


# ── SoN Score: normalized sum (SoN PRD v2.0) ─────────────────────────────
# Full formula (4 dims): (Sigma - 4) / 16 * 10
# Partial (<4 dims):     (Sigma - n) / (n*4) * 10,  flagged as partial

valid_dims = {d: s for d, s in dim_scores.items() if s is not None}
n_valid    = len(valid_dims)

if n_valid >= 1:
    total     = sum(valid_dims.values())
    son_score = (total - n_valid) / (n_valid * 4) * 10
    partial   = n_valid < 4
else:
    son_score = None
    partial   = True

def son_concern_label(score):
    if score is None: return "Insufficient data"
    if score < 4: return "Very Low"
    if score < 5: return "Low"
    if score < 7: return "Moderate"
    if score < 8: return "High"
    return "Very High"


# ── Dimension summary ─────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"DIMENSION SCORES  (TNFD Annex 2 | Protocol B v1.0)")
print(f"{'='*72}")
print(f"  {'Dim':<5} {'Dimension':<30} {'Score':<7} {'Concern':<15} Coverage")
print("  " + "-"*70)

for dim_num in [1, 2, 3, 4]:
    dname  = DIM_NAMES[dim_num]
    score  = dim_scores.get(dim_num)
    n_pop  = len(dim_populated.get(dim_num, []))
    n_tot  = len(DIM_MAP[dim_num])
    s_str  = f"{score:.2f}" if score is not None else "—"
    label  = dim_label(score) if score is not None else "No data"
    flag   = "  ← in-situ metrics missing" if n_pop < n_tot else ""
    print(f"  {dim_num:<5} {dname:<30} {s_str:<7} {label:<15} {n_pop}/{n_tot}{flag}")


# ── SoN Score ────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"STATE OF NATURE SCORE  (SoN PRD v2.0)")
print(f"{'='*72}")

if son_score is not None:
    print(f"  SoN Score:   {son_score:.1f} / 10")
    print(f"  Concern:     {son_concern_label(son_score)}")
    print(f"  Dims used:   {n_valid} of 4")
    if partial:
        missing = [DIM_NAMES[d] for d in [1,2,3,4] if dim_scores.get(d) is None]
        print(f"  Note: Partial score — no computable data for: {', '.join(missing)}")
        print(f"        Add in-situ species richness to complete Dim 3.")
    print()
    print(f"  Formula: ({total:.2f} - {n_valid}) / ({n_valid} x 4) x 10 = {son_score:.1f}")
    print()
    print(f"  Protocol B v1.0: >=85%=VL | 70-84%=L | 50-69%=M | 30-49%=H | <30%=VH")
    print(f"  Basis: Newbold et al. (2016) + GLOBIO4/SBTN literature")
    print(f"  Review trigger: >=10 Darukaa sites across >=3 ecoregion types")
else:
    print(f"  No dimension has computable data — check pipeline output.")



DIMENSION SCORES  (TNFD Annex 2 | Protocol B v1.0)
  Dim   Dimension                      Score   Concern         Coverage
  ----------------------------------------------------------------------
  1     Ecosystem Extent               1.80    Low             5/5
  2     Ecosystem Condition            2.00    Low             10/10
  3     Species Population Size        3.00    Moderate        2/2
  4     Species Extinction Risk        1.50    Very Low        2/3  ← in-situ metrics missing

STATE OF NATURE SCORE  (SoN PRD v2.0)
  SoN Score:   2.7 / 10
  Concern:     Very Low
  Dims used:   4 of 4

  Formula: (8.30 - 4) / (4 x 4) x 10 = 2.7

  Protocol B v1.0: >=85%=VL | 70-84%=L | 50-69%=M | 30-49%=H | <30%=VH
  Basis: Newbold et al. (2016) + GLOBIO4/SBTN literature
  Review trigger: >=10 Darukaa sites across >=3 ecoregion types


In [31]:
# ── Dimension concern profile (count of indicators per concern level) ─────
print(f"\n{'='*70}")
print(f"DIMENSION CONCERN PROFILES")
print(f"(count of indicators at each concern level within each dimension)")
print(f"{'='*70}")
print(f"  This is the primary output for TNFD LEAP Step 3 disclosure.")
print(f"  Concerns: VL=Very Low · L=Low · M=Moderate · H=High · VH=Very High\n")

labels_order = ["VL", "L", "M", "H", "VH", "—"]

for dim_num, indicators in DIM_MAP.items():
    counts = {l: [] for l in labels_order}
    for name in indicators:
        mc = metric_concerns.get(name)
        cl = mc["concern_label"] if mc else "—"
        if cl not in counts:
            cl = "—"
        counts[cl].append(mc["display_name"] if mc else name)

    print(f"  Dim {dim_num} — {DIM_NAMES[dim_num]}")
    for level in labels_order:
        items = counts[level]
        if not items: continue
        indicator_names = ", ".join(items)
        print(f"    {level:>2}: {len(items)} — {indicator_names}")
    print()


DIMENSION CONCERN PROFILES
(count of indicators at each concern level within each dimension)
  This is the primary output for TNFD LEAP Step 3 disclosure.
  Concerns: VL=Very Low · L=Low · M=Moderate · H=High · VH=Very High

  Dim 1 — Ecosystem Extent
    VL: 4 — Natural Habitat Extent, Natural Land Cover Proportion, Landscape Connectivity (CPLAND), Habitat Loss Rate
    VH: 1 — KBA/IBA Overlap

  Dim 2 — Ecosystem Condition
    VL: 5 — Ecosystem Integrity Index, EII: Structural Integrity, EII: Compositional Integrity, EII: Functional Integrity, Biodiversity Intactness Index
     L: 3 — Vegetation Structure (NDVI), Forest Landscape Integrity Index, Aridity Index
     H: 1 — Potentially Disappeared Fraction
    VH: 1 — Habitat Health Index (HHI)

  Dim 3 — Species Population Size
    VL: 1 — Flagship Habitat Viability
    VH: 1 — Endemic Species Richness

  Dim 4 — Species Extinction Risk
    VL: 1 — Threatened Species Richness
     L: 1 — Composite Extinction-Risk Index
     —: 1 — ST

## 8. Download results

In [32]:
import json as _json
from google.colab import files

# Augment the report with SoN scoring output
son_output = {
    "metric_concerns": metric_concerns,
    "dim_scores": {
        str(d): {
            "score":       dim_scores.get(d),
            "concern_label": dim_label(dim_scores.get(d)),
            "n_populated": len(dim_populated.get(d, [])),
            "n_total":     len(DIM_MAP[d]),
        }
        for d in [1, 2, 3, 4]
    },
    "son_score":   son_score,
    "son_concern": son_concern_label(son_score),
    "n_dims_used": n_valid,
    "partial_score": partial,
    "threshold_protocol": "Protocol A (published) + Protocol B v1.0 (fixed, Newbold 2016 + GLOBIO4/SBTN basis)",
    "formula": "SoN = (Sigma dim_scores - n_dims) / (n_dims x 4) x 10",
}

combined_path = f"{OUTPUT_DIR}/benchmark_scorecard_with_son.json"
with open(combined_path, "w") as _f:
    _json.dump({**report, "son_scoring": son_output}, _f, indent=2, default=str)

print("Downloading results...")
files.download(f"{OUTPUT_DIR}/benchmark_scorecard.json")
files.download(f"{OUTPUT_DIR}/benchmark_scorecard.csv")
files.download(combined_path)

print("\n\u2713 Downloaded:")
print("  benchmark_scorecard.json          — raw intactness ratios")
print("  benchmark_scorecard.csv           — same, CSV format")
print("  benchmark_scorecard_with_son.json — includes SoN concern levels & score")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Downloaded:
  benchmark_scorecard.json          — raw intactness ratios
  benchmark_scorecard.csv           — same, CSV format
  benchmark_scorecard_with_son.json — includes SoN concern levels & score


---

## (Optional) Add a custom indicator

Adding any new indicator requires only a few lines — no pipeline changes needed.

In [ ]:
# Example: Register a custom indicator

def extract_my_indicator(geometry, config):
    """
    Your extraction logic here.
    Return dict with 'value' (float) and 'pixels' (array or None).
    """
    return {"value": 0.42, "pixels": None}

registry.register(
    name="my_indicator",
    display_name="My Custom Indicator",
    source_type="gee",   # or "local_raster", "api", "in_situ"
    extract_fn=extract_my_indicator,
    unit="index",
    value_range=(0.0, 1.0),
    citation="Author et al. (year). DOI:...",
    tier2_eligible=True,   # False for threat/pressure indicators
    higher_is_better=True,
    reference_radius_km=50.0,
    pillar=2,              # TNFD dim: 1=Extent, 2=Condition, 3=Population, 4=Extinction
)

print(f"✓ Registry now has {len(registry)} indicators")
print(f"  Tier 2 eligible: {len(registry.tier2_indicators())}")